<a href="https://colab.research.google.com/github/Tejaswimadastu/Deep_Learning/blob/main/GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build a Mini GPT
### Smart Next Word Prediction System using GPT Architecture

Here implements a simplified GPT-style Decoder-Only Transformer step-by-step with explanations for every task

### Task 1: Data Preparation

Objective

Prepare text data for GPT training.

Load dataset
Convert to lowercase
Remove unwanted symbols
Tokenize text
Build vocabulary
Create input-output sequences

Example: Input: 'the cat chased' Target: 'the'

In [1]:
texts = [
    "The cat chased the mouse.",
    "The dog barked loudly.",
    "Machine learning is powerful.",
    "Deep learning uses neural networks."
]

print(texts)

['The cat chased the mouse.', 'The dog barked loudly.', 'Machine learning is powerful.', 'Deep learning uses neural networks.']


In [2]:
lowercase_texts = [text.lower() for text in texts]

print(lowercase_texts)

['the cat chased the mouse.', 'the dog barked loudly.', 'machine learning is powerful.', 'deep learning uses neural networks.']


In [3]:
import re

cleaned_texts = [re.sub(r'[^a-z\s]', '', text) for text in lowercase_texts]

print(cleaned_texts)

['the cat chased the mouse', 'the dog barked loudly', 'machine learning is powerful', 'deep learning uses neural networks']


In [4]:
tokenized_texts = [text.split() for text in cleaned_texts]

print(tokenized_texts)

[['the', 'cat', 'chased', 'the', 'mouse'], ['the', 'dog', 'barked', 'loudly'], ['machine', 'learning', 'is', 'powerful'], ['deep', 'learning', 'uses', 'neural', 'networks']]


In [5]:
vocab = set()

for tokens in tokenized_texts:
    vocab.update(tokens)

vocab = sorted(vocab)

print(vocab)

['barked', 'cat', 'chased', 'deep', 'dog', 'is', 'learning', 'loudly', 'machine', 'mouse', 'networks', 'neural', 'powerful', 'the', 'uses']


In [6]:
word_to_index = {word: idx for idx, word in enumerate(vocab)}

print(word_to_index)

{'barked': 0, 'cat': 1, 'chased': 2, 'deep': 3, 'dog': 4, 'is': 5, 'learning': 6, 'loudly': 7, 'machine': 8, 'mouse': 9, 'networks': 10, 'neural': 11, 'powerful': 12, 'the': 13, 'uses': 14}


In [7]:
tokens = tokenized_texts[0]
print(tokens)

['the', 'cat', 'chased', 'the', 'mouse']


In [8]:
for i in range(1, len(tokens)):
    input_seq = tokens[:i]
    target = tokens[i]

    print("Input:", input_seq, "Target:", target)

Input: ['the'] Target: cat
Input: ['the', 'cat'] Target: chased
Input: ['the', 'cat', 'chased'] Target: the
Input: ['the', 'cat', 'chased', 'the'] Target: mouse


### Task 2: Token Embeddings
Objective
Convert token IDs into dense vectors.

Example:

cat → [0.12,0.55,0.34]
dog → [0.21,0.77,0.61]


In [9]:
word_to_index = {
    'cat': 0,
    'dog': 1,
    'mouse': 2
}

In [10]:
embedding_dim = 3

In [11]:
import numpy as np

embedding_matrix = np.random.rand(3, 3)

print(embedding_matrix)

[[0.44004637 0.28247491 0.03261563]
 [0.13479176 0.1266147  0.10306141]
 [0.15277941 0.55268112 0.22953133]]


In [12]:
cat_id = word_to_index['cat']

cat_embedding = embedding_matrix[cat_id]

print(cat_embedding)

[0.44004637 0.28247491 0.03261563]


In [13]:
dog_id = word_to_index['dog']

dog_embedding = embedding_matrix[dog_id]

print(dog_embedding)

[0.13479176 0.1266147  0.10306141]


### Using PyTorch (GPT Style)

In [14]:
import torch
import torch.nn as nn

embedding = nn.Embedding(num_embeddings=3, embedding_dim=3)

token_id = torch.tensor([0])

print(embedding(token_id))

tensor([[-0.1731, -1.1280, -1.0475]], grad_fn=<EmbeddingBackward0>)


### Task 3: Positional Encoding
Objective

Add positional information.

Example:

the → position 0

cat → position 1

chased → position 2

In [15]:
tokens = ['the', 'cat', 'chased']

In [18]:
for pos, token in enumerate(tokens):
    print(token, "→ position", pos)

the → position 0
cat → position 1
chased → position 2


### Task 4: Masked Self Attention (Causal Attention)

In [19]:
import torch
import torch.nn.functional as F

tokens = ["the", "cat", "chased", "mouse"]
seq_len = len(tokens)

# Random attention scores
scores = torch.randn(seq_len, seq_len)

print("Attention Scores:")
print(scores)

# Causal Mask
mask = torch.tril(torch.ones(seq_len, seq_len))

print("\nAttention Mask:")
print(mask)

# Apply Mask
masked_scores = scores.masked_fill(mask == 0, float('-inf'))

# Softmax -> Attention Weights
attention_weights = F.softmax(masked_scores, dim=-1)

print("\nAttention Weights:")
print(attention_weights)

Attention Scores:
tensor([[ 0.6726, -0.0283,  0.4521,  0.6787],
        [-0.5188, -0.2912,  1.0010,  1.5708],
        [-0.7029,  0.0703, -0.7104,  0.1109],
        [ 0.1132, -0.9784,  0.1194, -0.7537]])

Attention Mask:
tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])

Attention Weights:
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.4433, 0.5567, 0.0000, 0.0000],
        [0.2404, 0.5209, 0.2386, 0.0000],
        [0.3621, 0.1215, 0.3643, 0.1521]])


### Task 5: Multi-Head Attention (4 Heads)

In [20]:
import torch
import torch.nn as nn

# Example embeddings
batch_size = 1
seq_len = 4
embed_dim = 16

x = torch.randn(batch_size, seq_len, embed_dim)

# 4 Attention Heads
multihead_attn = nn.MultiheadAttention(
    embed_dim=16,
    num_heads=4,
    batch_first=True
)

output, attention_weights = multihead_attn(
    x, x, x
)

print("Output Shape:")
print(output.shape)

print("\nAttention Weights Shape:")
print(attention_weights.shape)

print("\nAttention Weights:")
print(attention_weights)

Output Shape:
torch.Size([1, 4, 16])

Attention Weights Shape:
torch.Size([1, 4, 4])

Attention Weights:
tensor([[[0.2699, 0.2162, 0.2434, 0.2705],
         [0.2243, 0.2945, 0.2562, 0.2249],
         [0.2445, 0.2427, 0.2885, 0.2243],
         [0.2519, 0.2522, 0.2323, 0.2636]]], grad_fn=<MeanBackward1>)


In [21]:
heads = {
    "Head 1": "Grammar",
    "Head 2": "Context",
    "Head 3": "Long Dependencies",
    "Head 4": "Semantics"
}

for head, purpose in heads.items():
    print(f"{head} -> {purpose}")

Head 1 -> Grammar
Head 2 -> Context
Head 3 -> Long Dependencies
Head 4 -> Semantics


### Task 6: GPT Decoder Block
Architecture:
Masked Multi Head Attention
        ↓
Add & Normalize
        ↓
Feed Forward Network
        ↓
Add & Normalize


In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [23]:
class MaskedMultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()

        self.attention = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

    def forward(self, x):
        seq_len = x.size(1)

        mask = torch.triu(
            torch.ones(seq_len, seq_len),
            diagonal=1
        ).bool()

        output, weights = self.attention(
            x, x, x,
            attn_mask=mask
        )

        return output

In [24]:
class FeedForward(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()

        self.fc1 = nn.Linear(embed_dim, 4 * embed_dim)
        self.fc2 = nn.Linear(4 * embed_dim, embed_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [25]:
class GPTDecoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()

        self.attention = MaskedMultiHeadAttention(
            embed_dim,
            num_heads
        )

        self.norm1 = nn.LayerNorm(embed_dim)

        self.ffn = FeedForward(embed_dim)

        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):

        attn_output = self.attention(x)

        x = self.norm1(x + attn_output)

        ffn_output = self.ffn(x)

        x = self.norm2(x + ffn_output)

        return x

In [26]:
embed_dim = 16
num_heads = 4

decoder = GPTDecoderBlock(
    embed_dim,
    num_heads
)

x = torch.randn(1, 5, embed_dim)

output = decoder(x)

print("Input Shape :", x.shape)
print("Output Shape:", output.shape)

Input Shape : torch.Size([1, 5, 16])
Output Shape: torch.Size([1, 5, 16])


### Task 7: GPT Model
Architecture:
Input Tokens
      ↓
Embedding
      ↓
Position Encoding
      ↓
Decoder Block
      ↓
Decoder Block
      ↓
Linear
      ↓
Softmax

In [27]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [28]:
class PositionalEncoding(nn.Module):
    def __init__(self, max_len, embed_dim):
        super().__init__()

        self.embedding = nn.Embedding(
            max_len,
            embed_dim
        )

    def forward(self, x):
        seq_len = x.size(1)

        positions = torch.arange(
            seq_len,
            device=x.device
        )

        positions = positions.unsqueeze(0)

        return self.embedding(positions)

In [29]:
class FeedForward(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()

        self.fc1 = nn.Linear(embed_dim, 4 * embed_dim)
        self.fc2 = nn.Linear(4 * embed_dim, embed_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [30]:
class GPTDecoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()

        self.attention = nn.MultiheadAttention(
            embed_dim,
            num_heads,
            batch_first=True
        )

        self.norm1 = nn.LayerNorm(embed_dim)

        self.ffn = FeedForward(embed_dim)

        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):

        seq_len = x.size(1)

        mask = torch.triu(
            torch.ones(seq_len, seq_len),
            diagonal=1
        ).bool().to(x.device)

        attn_output, _ = self.attention(
            x, x, x,
            attn_mask=mask
        )

        x = self.norm1(x + attn_output)

        ffn_output = self.ffn(x)

        x = self.norm2(x + ffn_output)

        return x

In [31]:
class GPTModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim,
        num_heads,
        max_len
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            embed_dim
        )

        self.position_encoding = PositionalEncoding(
            max_len,
            embed_dim
        )

        self.decoder1 = GPTDecoderBlock(
            embed_dim,
            num_heads
        )

        self.decoder2 = GPTDecoderBlock(
            embed_dim,
            num_heads
        )

        self.linear = nn.Linear(
            embed_dim,
            vocab_size
        )

    def forward(self, x):

        token_embed = self.token_embedding(x)

        pos_embed = self.position_encoding(x)

        x = token_embed + pos_embed

        x = self.decoder1(x)

        x = self.decoder2(x)

        logits = self.linear(x)

        probabilities = F.softmax(
            logits,
            dim=-1
        )

        return probabilities

In [32]:
vocab_size = 15
embed_dim = 16
num_heads = 4
max_len = 20

model = GPTModel(
    vocab_size,
    embed_dim,
    num_heads,
    max_len
)

In [33]:
input_tokens = torch.tensor([
    [1, 2, 3]
])

In [34]:
output = model(input_tokens)

print("Output Shape:")
print(output.shape)

Output Shape:
torch.Size([1, 3, 15])


### Task 8: Next Token Prediction

In [35]:
sentence = "The cat chased"

In [36]:
input_tokens = torch.tensor([[1, 2, 3]])

output = model(input_tokens)

next_token = torch.argmax(
    output[:, -1, :],
    dim=-1
)

print("Predicted Token ID:")
print(next_token.item())

Predicted Token ID:
5


In [38]:
index_to_word = {idx: word for idx, word in enumerate(vocab)}

# Adding a <PAD> token at index 0 if it's not already in vocab, and shifting other indices if necessary.
# For now, let's assume the vocab starts from 0.
# If vocab contains all words, we can directly use it.
# Based on cell 7K3IDzzZlr_m, vocab starts with 'barked' at index 0
# We need to make sure index 0 is for '<PAD>' if that's the intention.
# However, looking at the previous word_to_index: {'barked': 0, 'cat': 1, ...}
# and input_tokens = torch.tensor([[1, 2, 3]]) meaning 'the', 'cat', 'chased'
# It seems that the earlier word_to_index did not include <PAD>.
# Let's align index_to_word with the existing word_to_index to avoid confusion.

# Re-create word_to_index from vocab to ensure consistency with the model's vocabulary
word_to_index_full = {word: idx for idx, word in enumerate(vocab)}
index_to_word = {idx: word for word, idx in word_to_index_full.items()}

predicted_word = index_to_word[next_token.item()]

print(predicted_word)

is


### Task 9: Text Generation


In [39]:
def generate_text(model, input_tokens, max_new_tokens):

    generated = input_tokens

    for _ in range(max_new_tokens):

        output = model(generated)

        next_token = torch.argmax(
            output[:, -1, :],
            dim=-1,
            keepdim=True
        )

        generated = torch.cat(
            [generated, next_token],
            dim=1
        )

    return generated

In [40]:
prompt = torch.tensor([[5, 6]])

generated_tokens = generate_text(
    model,
    prompt,
    max_new_tokens=5
)

print(generated_tokens)

tensor([[ 5,  6, 11,  2, 11, 10,  8]])


In [41]:
import torch

def generate_text(model, input_tokens, max_new_tokens):

    generated = input_tokens

    count = 0

    while count < max_new_tokens:

        output = model(generated)

        next_token = torch.argmax(
            output[:, -1, :],
            dim=-1,
            keepdim=True
        )

        generated = torch.cat(
            [generated, next_token],
            dim=1
        )

        count += 1

    return generated

In [42]:
word_to_index = {
    "machine": 0,
    "learning": 1,
    "is": 2,
    "powerful": 3,
    "because": 4
}

index_to_word = {v:k for k,v in word_to_index.items()}

In [43]:
prompt = torch.tensor([[0, 1]])  # machine learning

In [44]:
generated_tokens = generate_text(
    model,
    prompt,
    max_new_tokens=3
)

print(generated_tokens)

tensor([[ 0,  1,  8,  8, 11]])


In [46]:
generated_tokens = generate_text(
    model,
    prompt,
    max_new_tokens=3
)

# Recreate index_to_word from the full vocabulary mapping (word_to_index_full)
# to ensure all generated tokens can be mapped.
# word_to_index_full is a global variable from an earlier cell.
index_to_word_full = {idx: word for word, idx in word_to_index_full.items()}

generated_words = []

for token in generated_tokens[0]:
    generated_words.append(index_to_word_full[token.item()])

generated_text = " ".join(generated_words)

# Dynamically get the initial prompt words from the prompt tensor
initial_prompt_words = [index_to_word_full[t.item()] for t in prompt[0]]
input_prompt_text = " ".join(initial_prompt_words)

print("Input Prompt :", input_prompt_text)
print("Generated Text:", generated_text)

Input Prompt : barked cat
Generated Text: barked cat machine machine neural


In [47]:
print("=" * 40)
print("GPT Text Generation")
print("=" * 40)

print(f"Input Prompt : {input_prompt_text}")
print()
print("Generated Text:")
print(generated_text)

print("=" * 40)

GPT Text Generation
Input Prompt : barked cat

Generated Text:
barked cat machine machine neural


In [50]:
user_input = input("Enter a prompt: ")

words = user_input.lower().split()

prompt_ids = []

for word in words:
    if word in word_to_index_full:
        prompt_ids.append(word_to_index_full[word])

if len(prompt_ids) == 0:
    print("No valid words found in vocabulary.")
else:
    prompt = torch.tensor([prompt_ids])

    generated_tokens = generate_text(
        model,
        prompt,
        max_new_tokens=5
    )

    generated_words = []

    for token in generated_tokens[0]:
        generated_words.append(
            index_to_word_full[token.item()]
        )

    generated_text = " ".join(generated_words)

    print("\nGenerated Text:")
    print(generated_text)

Enter a prompt: Deep learning

Generated Text:
deep learning neural chased neural networks machine
